In [1]:
!pip install pyfoma

  Using cached pyfoma-1.0.2-py3-none-any.whl (32 kB)
  Using cached graphviz-0.20.1-py3-none-any.whl (47 kB)


In [ ]:
#spanish plural noun grammar

In [2]:
from pyfoma import *
fsts = {}

fsts['nouns'] = FST.re("alumno|amigo|bebé|chica|composición|doctor|flor|interés|lápiz|nación|papel|pez|presidente|rey|tapiz|tisú")

In [3]:
fsts['suffixes'] = FST.re("( '[N]' '[Sg]' ):'' | ( '[N]' '[Pl]' ):(\+s)")

In [4]:
fsts['lexicon'] = FST.re("$nouns $suffixes", fsts)

In [5]:
print(Paradigm(fsts['lexicon'], ".*"))

alumno       [N][Pl]  alumno+s       
alumno       [N][Sg]  alumno         
amigo        [N][Pl]  amigo+s        
amigo        [N][Sg]  amigo          
bebé         [N][Pl]  bebé+s         
bebé         [N][Sg]  bebé           
chica        [N][Pl]  chica+s        
chica        [N][Sg]  chica          
composición  [N][Pl]  composición+s  
composición  [N][Sg]  composición    
doctor       [N][Pl]  doctor+s       
doctor       [N][Sg]  doctor         
flor         [N][Pl]  flor+s         
flor         [N][Sg]  flor           
interés      [N][Pl]  interés+s      
interés      [N][Sg]  interés        
lápiz        [N][Pl]  lápiz+s        
lápiz        [N][Sg]  lápiz          
nación       [N][Pl]  nación+s       
nación       [N][Sg]  nación         
papel        [N][Pl]  papel+s        
papel        [N][Sg]  papel          
pez          [N][Pl]  pez+s          
pez          [N][Sg]  pez            
presidente   [N][Pl]  presidente+s   
presidente   [N][Sg]  presidente     
rey         

Rewrite rules
1. if ends in constonant, add es
2. if ends in í or ú in, you can add -s or -es to pluralize the word
3. -- 7: Words ending in n or s that stress the final syllable (stressed vowel) in the singular lose the written accent in the plural form.
8. if ends in z in singular, change z to c in plural
9. if ends in vowel, add s


In [6]:
fsts['unstress'] = FST.re("[aeiou]")
fsts['stress'] = FST.re("[áéíóú]")
fsts['con'] = FST.re("[bcdfghjklmnpqrstvwxyz]")

In [7]:
#if ends in constonant, add -es
fsts['rule1'] = FST.re("$^rewrite('':e / $con _ '+' s)", fsts)
#insert an e between a constant and '+s'

#if ends in í or ú, you can add -s or -es to pluralize the word -spanishdict
fsts['rule2'] = FST.re("$^rewrite('':e / í|ú _ '+'s)")

#Words ending in n or s that stress the final syllable in the singular lose the written accent in the plural form.
#fsts['rule3'] = FST.re("$^rewrite((($stress):($unstress)) /_n|s)", fsts) #this doesn't work because it replaces the stress vowel with every unstress

#the rules below are doing that ^ for every stressed vowel that is followed by n or s and pluralized
fsts['rule3'] = FST.re("$^rewrite(á:a / _ne'+'s|se'+'s)")
fsts['rule4'] = FST.re("$^rewrite(é:e / _ne'+'s|se'+'s)")
fsts['rule5'] = FST.re("$^rewrite(í:i / _ne'+'s|se'+'s)")
fsts['rule6'] = FST.re("$^rewrite(ó:o / _ne'+'s|se'+'s)")
fsts['rule7'] = FST.re("$^rewrite(ú:u / _ne'+'s|se'+'s)")

#if ends in z, change z to c
fsts['rule8'] = FST.re("$^rewrite((z:c) / _ e'+' s)")

#if ends in vowel, add s
fsts['cleanup'] = FST.re("$^rewrite('+':'')")

grammar = FST.re("$lexicon @$rule1 @$rule2 @$rule3 @$rule4 @$rule5 @$rule6 @$rule7 @$rule8 @$cleanup", fsts)
print(Paradigm(grammar, ".*"))

alumno       [N][Pl]  alumnos        
alumno       [N][Sg]  alumno         
amigo        [N][Pl]  amigos         
amigo        [N][Sg]  amigo          
bebé         [N][Pl]  bebés          
bebé         [N][Sg]  bebé           
chica        [N][Pl]  chicas         
chica        [N][Sg]  chica          
composición  [N][Pl]  composiciones  
composición  [N][Sg]  composición    
doctor       [N][Pl]  doctores       
doctor       [N][Sg]  doctor         
flor         [N][Pl]  flores         
flor         [N][Sg]  flor           
interés      [N][Pl]  intereses      
interés      [N][Sg]  interés        
lápiz        [N][Pl]  lápices        
lápiz        [N][Sg]  lápiz          
nación       [N][Pl]  naciones       
nación       [N][Sg]  nación         
papel        [N][Pl]  papeles        
papel        [N][Sg]  papel          
pez          [N][Pl]  peces          
pez          [N][Sg]  pez            
presidente   [N][Pl]  presidentes    
presidente   [N][Sg]  presidente     
rey         

Correct Pluralization 

alumnos      
amigos      
bebés        
chicas      
composiciones \
doctores    
flores      
intereses \
lápices      
naciones    
papeles      
peces        
presidentes  
reyes        
tapices      
tisúes \
Rule: If a singular noun ends in í or ú, you can add -s or -es to pluralize the word. The -es plural form is considered to be a bit fancier. - spanishdict
      